# OpenAQ API Exploration

This notebook is used to inspect OpenAQ API responses and validate ingestion logic before implementation in the production pipeline.

Exploration performed here is used to understand source schemas, pagination, date ranges, sensor behavior, and API edge cases. Production logic is implemented separately under `ingestion/` and verified with automated tests.

## Environment Verification

Verify that the notebook is using the project's virtual environment. This ensures exploratory code runs with the same Python dependencies used by the application.

In [1]:
import sys
from pathlib import Path
import json
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from ingestion.client import OpenAQClient

client = OpenAQClient()
payload = client.get("countries", {"page": 1, "limit": 5})

print("Results:", len(payload["results"]))
print("Pagination metadata:", payload.get("meta"))
print("First country:", payload["results"][0])

Results: 5
Pagination metadata: {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 5, 'found': 159}
First country: {'id': 1, 'code': 'ID', 'name': 'Indonesia', 'datetimeFirst': '2016-01-30T01:00:00Z', 'datetimeLast': '2026-10-22T14:58:00Z', 'parameters': [{'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': None}, {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None}, {'id': 3, 'name': 'o3', 'units': 'µg/m³', 'displayName': None}, {'id': 10, 'name': 'o3', 'units': 'ppm', 'displayName': None}, {'id': 11, 'name': 'bc', 'units': 'µg/m³', 'displayName': None}, {'id': 15, 'name': 'no2', 'units': 'ppb', 'displayName': None}, {'id': 19, 'name': 'pm1', 'units': 'µg/m³', 'displayName': None}, {'id': 21, 'name': 'co2', 'units': 'ppm', 'displayName': None}, {'id': 22, 'name': 'wind_direction', 'units': 'deg', 'displayName': None}, {'id': 23, 'name': 'nox', 'units': 'ppb', 'displayName': None}, {'id': 24, 'name': 'no', 'units': 'ppb', 'displayName': None}, {'id': 34, 'name

## Raw ingestion verification
Fetch one OpenAQ API page, write the unmodified JSON response to the Azure Raw container, and verify the stored content against its checksum. This is a controlled development test, not a historical backfill.

In [3]:
import importlib
import ingestion.storage

importlib.reload(ingestion.storage)

from ingestion.storage import RawWriter

## Validate Azure Raw filesystem
Confirm that Databricks-independent Python ingestion can access the configured ADLS Gen2 filesystem and create partition directories. This isolates storage configuration from API ingestion logic.

In [4]:
from config.settings import AZURE_RAW_FILE_SYSTEM, AZURE_STORAGE_ACCOUNT_NAME
from ingestion.storage import RawWriter

print("Storage account:", AZURE_STORAGE_ACCOUNT_NAME)
print("Filesystem:", AZURE_RAW_FILE_SYSTEM)

writer = RawWriter()

print("Filesystem exists:", writer.filesystem.exists())

writer._ensure_directory("countries/ingestion_date=2026-10-18")

print("Directory creation: PASSED")

Storage account: openaqdataplatform
Filesystem: raw


DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' pa

Filesystem exists: True
Directory creation: PASSED


In [5]:
from datetime import datetime, timezone
from uuid import uuid4

ingestion_date = datetime.now(timezone.utc).date()
directory_path = f"countries/ingestion_date={ingestion_date}"

directory = writer.filesystem.get_directory_client(directory_path)

print("Storage account:", writer.filesystem.account_name)
print("Directory:", directory_path)
print("Directory exists:", directory.exists())

Storage account: openaqdataplatform
Directory: countries/ingestion_date=2026-10-22
Directory exists: True


In [6]:
from uuid import uuid4

test_path = f"{directory_path}/upload_test_{uuid4().hex}.txt"
file_client = writer.filesystem.get_file_client(test_path)

data = b"ADLS upload verification"

file_client.create_file()  # Explicitly create the file before writing.
file_client.append_data(data=data, offset=0, length=len(data))
file_client.flush_data(len(data))  # Commit the uploaded bytes.

stored_data = file_client.download_file().readall()

assert stored_data == data  # Verify the persisted content matches the source.

print("File creation: PASSED")
print("File upload: PASSED")
print("Read-back verification: PASSED")
print("Uploaded path:", test_path)

File creation: PASSED
File upload: PASSED
Read-back verification: PASSED
Uploaded path: countries/ingestion_date=2026-10-22/upload_test_20c4d791887b4605b43d4fdac019ac80.txt


In [7]:
from datetime import datetime, timezone

from ingestion.client import OpenAQClient
from ingestion.storage import RawWriter

client = OpenAQClient()
writer = RawWriter()

payload = client.get("countries", {"page": 1, "limit": 5})
ingestion_date = datetime.now(timezone.utc).date()

path = writer.write_json(
    payload,
    f"countries/ingestion_date={ingestion_date}",
    "countries",
    "dev_raw_validation",
    "page_0001",
    {"page": 1}
)

stored_payload = writer.read_json(path)

assert stored_payload == payload  # Confirm the source JSON values survive landing.
assert writer.verify_json(path)  # Confirm the stored bytes match the recorded checksum.

print("Raw path:", path)
print("Records:", len(stored_payload["results"]))
print("Checksum verification: PASSED")

Raw path: countries/ingestion_date=2026-10-22/page_0001_20261022T151153469284Z_1100a2805558409d9cd7ea4f44768697.json
Records: 5
Checksum verification: PASSED


## Raw Backfill Reconciliation

### Validate that the records persisted in ADLS match the completed backfill checkpoint. Use the batch identifier to distinguish the original ingestion attempt from other files in the same partition.

In [8]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
checkpoints = json.loads((project_root / ".state/backfill.json").read_text())
checkpoint = next(item for item in checkpoints.values() if item["sensor_id"] == 12234787)

writer = RawWriter()
directory = "measurements/sensor_id=12234787/year=2026/month=09/day=01"

# Reconcile against the committed batch, not unrelated files in the partition.
metadata_files = [item.name for item in writer.filesystem.get_paths(path=directory) if item.name.endswith(".metadata.json")]
batch_files = [path for path in metadata_files if writer.read_json(path)["batch_id"] == checkpoint["batch_id"]]
source_files = [path.removesuffix(".metadata.json") + ".json" for path in batch_files]

# Count persisted observations rather than trusting the checkpoint alone.
persisted_count = sum(len(writer.read_json(path)["results"]) for path in source_files)

print("Checkpoint records:", checkpoint["record_count"])
print("Persisted records:", persisted_count)
print("Source files:", len(source_files))

assert persisted_count == checkpoint["record_count"], "Raw reconciliation failed"
print("Raw reconciliation: PASSED")

Checkpoint records: 88
Persisted records: 88
Source files: 1
Raw reconciliation: PASSED


### Backfill Idempotency Verification

Compare the completed checkpoint against persisted Raw files using the ingestion batch ID. Verify that the successful rerun did not create another set of files.

In [9]:
directory = "measurements/sensor_id=12234787/year=2026/month=09/day=01"

# Capture all persisted files in the partition, including lineage metadata.
files_before = {item.name for item in writer.filesystem.get_paths(path=directory)}
print("Files before rerun:", len(files_before))

Files before rerun: 4


In [17]:
files_after = {item.name for item in writer.filesystem.get_paths(path=directory)}

new_files = files_after - files_before
removed_files = files_before - files_after

print("Files before:", len(files_before))
print("Files after:", len(files_after))
print("New files:", len(new_files))
print("Removed files:", len(removed_files))

assert files_before == files_after, "Backfill idempotency failed"
print("Backfill idempotency: PASSED")

Files before: 4
Files after: 4
New files: 0
Removed files: 0
Backfill idempotency: PASSED


## OpenAQ Client Basic Request Validation

Validate the minimum HTTP request required to communicate with OpenAQ using the project's configured base URL and API key.

This establishes the request pattern that will later be moved into the production API client. At this stage, the goal is only to verify authentication, connectivity, status handling, and the structure of the returned JSON.

In [10]:
import requests

from config.settings import OPENAQ_API_KEY, OPENAQ_BASE_URL

headers = {"X-API-Key": OPENAQ_API_KEY}

response = requests.get(
    f"{OPENAQ_BASE_URL}/locations",
    headers=headers,
    params={"limit": 5},
    timeout=30
)

response.raise_for_status()
payload = response.json()

### Response Inspection

Inspect the top-level response structure and pagination metadata without printing the full payload. This confirms the fields the production client can rely on when processing paginated responses.

In [11]:
payload.keys(), payload["meta"]


(dict_keys(['meta', 'results']),
 {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 5, 'found': '>5'})

## Pagination Behavior

Test consecutive OpenAQ pages with a small page size to confirm how pagination behaves in practice.

The production client should not depend on `meta.found` being numeric. Instead, pagination will continue while a full page is returned and stop when the API returns fewer records than the requested limit.

In [12]:
for page in range(1, 4):
    response = requests.get(
        f"{OPENAQ_BASE_URL}/locations",
        headers=headers,
        params={"limit": 5, "page": page},
        timeout=30,
    )
    response.raise_for_status()

    data = response.json()
    print(page, len(data["results"]), data["meta"])

1 5 {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 5, 'found': '>5'}
2 5 {'name': 'openaq-api', 'website': '/', 'page': 2, 'limit': 5, 'found': '>5'}
3 5 {'name': 'openaq-api', 'website': '/', 'page': 3, 'limit': 5, 'found': '>5'}


## Production Client Validation

Validate the production HTTP client against the OpenAQ API.

The client centralizes authentication, timeout handling, retries, and HTTP error handling so endpoint-specific ingestion logic does not duplicate networking concerns.

In [13]:
from ingestion.client import OpenAQClient

client = OpenAQClient()

payload = client.get(
    "locations",
    {"limit": 5, "page": 1},
)

payload["meta"], len(payload["results"])

({'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 5, 'found': '>5'},
 5)

## Production Pagination Validation

Validate that the production client retrieves consecutive pages while preserving the API's page boundaries.

The client yields one page at a time rather than accumulating the complete dataset in memory. This allows downstream ingestion to process and persist large measurement extracts incrementally.

In [14]:
pages = client.paginate(
    "locations",
    page_size=5,
)

for index, records in enumerate(pages, start=1):
    print(f"Page {index}: {len(records)} records")

    if index == 3:
        break

Page 1: 5 records
Page 2: 5 records
Page 3: 5 records


### Country Identifier Lookup

Retrieve India's OpenAQ country identifier required by the location endpoint. The API uses this identifier to scope location extraction to the selected country.

In [15]:
countries = client.get("countries", {"limit": 1000})

india = next(
    country
    for country in countries["results"]
    if country["name"] == "India"
)

india

INDIA_COUNTRY_ID = india["id"]
INDIA_COUNTRY_ID

9

## Location Extractor Validation

Validate that the location extractor applies the configured country filter and returns paginated records through the production OpenAQ client.

In [16]:
from ingestion.extract_locations import extract_locations

location_pages = extract_locations(
    client,
    country_id=INDIA_COUNTRY_ID,
)

first_page = next(location_pages)

len(first_page), first_page[0]["name"]

ModuleNotFoundError: No module named 'ingestion.extract_locations'

### Target Location Resolution

Resolve the OpenAQ location identifier for R K Puram, the monitoring site selected during source investigation for measurement ingestion.

In [ ]:
payload = client.get(
    "locations",
    {"name": "R K Puram", "limit": 100},
)

[
    (location["id"], location["name"])
    for location in payload["results"]
]

[(3, 'NMA - Nima'),
 (4, 'NMT - Nima'),
 (5, 'JTA - Jamestown'),
 (6, 'ADT - Asylum Down'),
 (7, 'ADEPA - Asylum Down'),
 (8, 'ADA - Asylum Down'),
 (9, 'ELC - East Legon'),
 (10, 'ELT - East Legon'),
 (11, 'ELA - East Legon'),
 (12, 'SPARTAN - IIT Kanpur'),
 (13, 'Delhi Technological University, Delhi - CPCB'),
 (14, 'SPARTAN - CITEDEF'),
 (15, 'IGI Airport'),
 (16, 'Civil Lines'),
 (17, 'R K Puram, Delhi - DPCC'),
 (18, 'SPARTAN - Vietnam Acad. Sci.'),
 (19, 'MNB'),
 (21, 'Beijing US Embassy'),
 (22, 'SPARTAN - Dhaka University'),
 (23, 'Amgalan'),
 (24, 'SPARTAN - NUS'),
 (25, "Parque O'Higgins"),
 (26, 'Inpesca'),
 (27, 'Concón'),
 (28, 'Ursynów'),
 (30, '100 ail'),
 (31, 'Wekerom-Riemterdijk'),
 (32, 'SPARTAN - Tsinghua University'),
 (33, 'Zaanstad-Hemkade'),
 (34, 'SPARTAN - Weizmann Institute'),
 (38, 'Cauquenes'),
 (39, 'Gobernación'),
 (41, 'Groningen-Europaweg'),
 (42, 'SPARTAN - Ilorin University'),
 (43, 'Konarskiego'),
 (44, 'Heerlen-Jamboreepad'),
 (45, 'Puente Alto'),
 

## Sensor Extractor Validation

Use the previously investigated R K Puram monitoring location to validate location-level sensor extraction.

This confirms the relationship between an OpenAQ location and its individual pollutant and meteorological sensors before measurement ingestion is implemented.

In [ ]:
from ingestion.extract_sensors import extract_sensors

sensors = extract_sensors(client, location_id=17)
[
    (
        sensor["id"],
        sensor["parameter"]["name"],
        sensor["parameter"]["units"],
    )
    for sensor in sensors
]

[(12234783, 'no', 'ppb'),
 (12234785, 'o3', 'µg/m³'),
 (12234787, 'pm25', 'µg/m³'),
 (12234784, 'no2', 'ppb'),
 (12234786, 'pm10', 'µg/m³'),
 (12234782, 'co', 'ppb'),
 (12234790, 'temperature', 'c'),
 (12234788, 'relativehumidity', '%'),
 (12234789, 'so2', 'ppb'),
 (14340715, 'wind_speed', 'm/s'),
 (14340714, 'wind_direction', 'deg'),
 (14340713, 'nox', 'ppb'),
 (399, 'pm10', 'µg/m³'),
 (35, 'pm25', 'µg/m³'),
 (394, 'o3', 'µg/m³'),
 (36, 'no2', 'µg/m³'),
 (392, 'co', 'µg/m³'),
 (393, 'so2', 'µg/m³')]

## Measurement Extractor Validation

Validate bounded measurement extraction using the current R K Puram PM2.5 sensor.

The request is restricted to a defined UTC time window and uses the production paginator, matching the extraction pattern required for historical backfills and incremental ingestion.

In [ ]:
from datetime import datetime, timezone

from ingestion.extract_measurements import extract_measurements

measurement_pages = extract_measurements(
    client,
    sensor_id=12234787,
    start=datetime(2026, 9, 1, tzinfo=timezone.utc),
    end=datetime(2026, 9, 2, tzinfo=timezone.utc)
)

measurements = next(measurement_pages)

len(measurements)

88

## Warehouse Source Model Investigation

Before expanding ingestion, we need to determine which OpenAQ resources are
required to support the dimensional warehouse model.

Target entities:

- Country
- Provider
- Owner
- Location
- Sensor
- Parameter
- Measurement

We will inspect real API responses first, map available source fields to the
target model, and only then decide which resources require production ingestion.

In [ ]:
from ingestion.client import OpenAQClient

# Reuse the tested API client so notebook exploration matches production API behavior
client = OpenAQClient()

### Inspect Location Metadata

Location metadata may already contain the country, provider and owner
relationships required by the warehouse.

We inspect a known location first to determine whether those entities can be
derived from the location resource or require separate source ingestion.

In [ ]:
# Use a previously investigated location so we can focus on metadata structure
location_payload = client.get("locations/17")

location_payload

{'meta': {'name': 'openaq-api',
  'website': '/',
  'page': 1,
  'limit': 100,
  'found': 1},
 'results': [{'id': 17,
   'name': 'R K Puram, Delhi - DPCC',
   'locality': None,
   'timezone': 'Asia/Kolkata',
   'country': {'id': 9, 'code': 'IN', 'name': 'India'},
   'owner': {'id': 8524, 'name': 'Delhi Pollution Control Committee'},
   'provider': {'id': 168, 'name': 'CPCB'},
   'isMobile': False,
   'isMonitor': True,
   'instruments': [{'id': 2, 'name': 'Government Monitor'},
    {'id': 2, 'name': 'Government Monitor'}],
   'sensors': [{'id': 12234782,
     'name': 'co ppb',
     'parameter': {'id': 102,
      'name': 'co',
      'units': 'ppb',
      'displayName': 'CO'}},
    {'id': 392,
     'name': 'co µg/m³',
     'parameter': {'id': 4,
      'name': 'co',
      'units': 'µg/m³',
      'displayName': 'CO mass'}},
    {'id': 12234783,
     'name': 'no ppb',
     'parameter': {'id': 24,
      'name': 'no',
      'units': 'ppb',
      'displayName': 'NO'}},
    {'id': 36,
     'nam

### Inspect Sensor Metadata

The location response already contains sensor IDs, names and parameter
information.

We now inspect the dedicated sensor resource to determine whether it provides
additional lifecycle, coverage or measurement metadata required by
`dim_sensor`, `dim_parameter` or downstream sensor-quality analysis.

This prevents us from creating a separate sensor ingestion pipeline unless the
additional source attributes justify it.

In [ ]:
# Use the known PM2.5 sensor from location 17 to inspect its complete metadata
SENSOR_ID = 12234787

sensor_payload = client.get(f"sensors/{SENSOR_ID}")

sensor_payload

{'meta': {'name': 'openaq-api',
  'website': '/',
  'page': 1,
  'limit': 100,
  'found': 1},
 'results': [{'id': 12234787,
   'name': 'pm25 µg/m³',
   'parameter': {'id': 2,
    'name': 'pm25',
    'units': 'µg/m³',
    'displayName': 'PM2.5'},
   'datetimeFirst': {'utc': '2025-02-18T20:15:00Z',
    'local': '2025-02-19T01:45:00+05:30'},
   'datetimeLast': {'utc': '2026-10-02T08:30:00Z',
    'local': '2026-10-02T14:00:00+05:30'},
   'coverage': {'expectedCount': 4,
    'expectedInterval': '01:00:00',
    'observedCount': 45090,
    'observedInterval': '11272:30:00',
    'percentComplete': 1127250.0,
    'percentCoverage': 1127250.0,
    'datetimeFrom': {'utc': '2025-02-18T20:15:00Z',
     'local': '2025-02-19T01:45:00+05:30'},
    'datetimeTo': {'utc': '2026-10-02T08:30:00Z',
     'local': '2026-10-02T14:00:00+05:30'}},
   'latest': {'datetime': {'utc': '2026-10-02T08:30:00Z',
     'local': '2026-10-02T14:00:00+05:30'},
    'value': 62.0,
    'coordinates': {'latitude': 28.563262, 'lo

### Inspect Parameter Metadata

Both location and sensor responses contain nested parameter information:
ID, name, unit and display name.

We now inspect the dedicated parameter resource to determine whether it
contains additional attributes required by `dim_parameter`.

If the dedicated resource adds useful descriptive metadata, it can become the
authoritative parameter source. Otherwise, parameter information may be
derived from the sensor metadata already being ingested.

In [ ]:
# Inspect the dedicated metadata for the PM2.5 parameter used by our known sensor
PARAMETER_ID = 2

parameter_payload = client.get(f"parameters/{PARAMETER_ID}")

parameter_payload


{'meta': {'name': 'openaq-api',
  'website': '/',
  'page': 1,
  'limit': 100,
  'found': 1},
 'results': [{'id': 2,
   'name': 'pm25',
   'units': 'µg/m³',
   'displayName': 'PM2.5',
   'description': 'Particulate matter less than 2.5 micrometers in diameter mass concentration'}]}

### Discover Candidate Countries

Before selecting monitoring locations, we first inspect the countries available
through OpenAQ.

At this stage we are only discovering potential geographic coverage. Country
selection will be followed by location and sensor-level checks so that the
final dataset supports meaningful cross-location analysis.

In [ ]:
# Retrieve a small country sample for geographic scope investigation
countries_payload = client.get(
    "countries",
    params={"limit": 100},
)

countries = countries_payload["results"]

print(f"Countries returned: {len(countries)}")
countries[:10]

Countries returned: 100


[{'id': 1,
  'code': 'ID',
  'name': 'Indonesia',
  'datetimeFirst': '2016-01-30T01:00:00Z',
  'datetimeLast': '2026-10-03T16:44:00Z',
  'parameters': [{'id': 1,
    'name': 'pm10',
    'units': 'µg/m³',
    'displayName': None},
   {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None},
   {'id': 3, 'name': 'o3', 'units': 'µg/m³', 'displayName': None},
   {'id': 10, 'name': 'o3', 'units': 'ppm', 'displayName': None},
   {'id': 11, 'name': 'bc', 'units': 'µg/m³', 'displayName': None},
   {'id': 15, 'name': 'no2', 'units': 'ppb', 'displayName': None},
   {'id': 19, 'name': 'pm1', 'units': 'µg/m³', 'displayName': None},
   {'id': 21, 'name': 'co2', 'units': 'ppm', 'displayName': None},
   {'id': 22, 'name': 'wind_direction', 'units': 'deg', 'displayName': None},
   {'id': 23, 'name': 'nox', 'units': 'ppb', 'displayName': None},
   {'id': 24, 'name': 'no', 'units': 'ppb', 'displayName': None},
   {'id': 34, 'name': 'wind_speed', 'units': 'm/s', 'displayName': None},
   {'id': 95

### Review Country Coverage

A tabular view makes it easier to compare the available country metadata
before deciding which countries deserve deeper location-level investigation.

This is exploratory notebook logic only; it is not yet part of the production
ingestion pipeline.

In [ ]:
import pandas as pd

# Flatten only the fields needed for country selection during exploration
countries_df = pd.DataFrame(
    [
        {
            "country_id": country["id"],
            "country_code": country["code"],
            "country_name": country["name"],
        }
        for country in countries
    ]
)

countries_df.head(20)

,country_id,country_code,country_name
0,1,ID,Indonesia
1,2,MY,Malaysia
2,3,CL,Chile
3,4,BO,Bolivia
4,5,PE,Peru
5,6,AR,Argentina
6,7,-99,Dhekelia
7,8,CY,Cyprus
8,9,IN,India
9,10,CN,China


### Evaluate Country-Level Data Coverage

The country resource includes observation dates and available parameters in
addition to country identity.

We use these fields to shortlist countries with recent data and useful
pollutant/environmental coverage before querying individual locations. This
reduces unnecessary API calls and makes geographic selection evidence-based.

Country-level dates describe overall source coverage and will only be used for
candidate selection, not as measurement-level completeness metrics.

**Source finding:** `parameters` can be null for some countries. During
exploration, null parameter collections are treated as empty collections so
countries without reported parameter metadata remain in the dataset.

In [ ]:
# Summarize country metadata needed for candidate selection
countries_df = pd.DataFrame(
    [
        {
            "country_id": country["id"],
            "country_code": country["code"],
            "country_name": country["name"],
            "datetime_first": country.get("datetimeFirst"),
            "datetime_last": country.get("datetimeLast"),
            "parameters": [
                parameter["name"]
                for parameter in (country.get("parameters") or [])
            ],
            "parameter_count": len(
                country.get("parameters") or []
            ),
        }
        for country in countries
    ]
)

countries_df.head()

,country_id,country_code,country_name,datetime_first,datetime_last,parameters,parameter_count
0,1,ID,Indonesia,2016-01-30T01:00:00Z,2026-10-03T16:44:00Z,"[pm10, pm25, o3, o3, bc, no2, pm1, co2, wind_d...",20
1,2,MY,Malaysia,2022-11-03T21:00:00Z,2026-10-03T16:00:00Z,"[pm10, pm25, pm1, relativehumidity, temperatur...",6
2,3,CL,Chile,2016-01-30T01:00:00Z,2026-10-03T16:00:00Z,"[pm10, pm25, o3, co, no2, so2, pm1, relativehu...",10
3,4,BO,Bolivia,2026-06-22T20:00:00Z,2026-10-03T16:00:00Z,"[pm25, pm1, relativehumidity, temperature, um003]",5
4,5,PE,Peru,2016-03-04T17:00:00Z,2026-10-03T16:00:00Z,"[pm10, pm25, o3, co, no2, so2, no2, co, so2, o...",17


### Shortlist Countries for Deeper Investigation

The planned analytical model requires comparable pollutants and, where
possible, meteorological measurements.

We therefore identify countries containing core air-quality parameters such
as PM2.5, PM10, NO2 and O3, while also recording whether temperature,
humidity and wind measurements are available.

This remains candidate discovery; no country is selected for production
ingestion at this stage.

In [ ]:
# Parameters most relevant to the planned analytical marts
CORE_POLLUTANTS = {"pm25", "pm10", "no2", "o3"}
WEATHER_PARAMETERS = {
    "temperature",
    "relativehumidity",
    "wind_speed",
    "wind_direction",
}


def summarize_parameter_coverage(parameters: list[str]) -> dict:
    available = set(parameters)

    return {
        "core_pollutants": len(available & CORE_POLLUTANTS),
        "has_pm25": "pm25" in available,
        "has_pm10": "pm10" in available,
        "has_no2": "no2" in available,
        "has_o3": "o3" in available,
        "weather_parameters": len(available & WEATHER_PARAMETERS),
    }


# Expand parameter availability into columns that are easier to compare
coverage_df = countries_df["parameters"].apply(
    summarize_parameter_coverage
).apply(pd.Series)

country_candidates_df = pd.concat(
    [countries_df, coverage_df],
    axis=1,
)

country_candidates_df[
    [
        "country_id",
        "country_code",
        "country_name",
        "datetime_last",
        "core_pollutants",
        "weather_parameters",
        "has_pm25",
        "has_pm10",
        "has_no2",
        "has_o3",
    ]
].sort_values(
    ["core_pollutants", "weather_parameters"],
    ascending=False,
).head(20)

,country_id,country_code,country_name,datetime_last,core_pollutants,weather_parameters,has_pm25,has_pm10,has_no2,has_o3
0,1,ID,Indonesia,2026-10-03T16:44:00Z,4,4,True,True,True,True
8,9,IN,India,2026-10-03T16:30:00Z,4,4,True,True,True,True
2,3,CL,Chile,2026-10-03T16:00:00Z,4,2,True,True,True,True
4,5,PE,Peru,2026-10-03T16:00:00Z,4,2,True,True,True,True
5,6,AR,Argentina,2026-10-04T03:00:00Z,4,2,True,True,True,True
7,8,CY,Cyprus,2026-10-03T16:00:00Z,4,2,True,True,True,True
9,10,CN,China,2026-10-03T16:00:00Z,4,2,True,True,True,True
10,11,IL,Israel,2026-10-03T16:00:00Z,4,2,True,True,True,True
18,22,FR,France,2026-10-03T16:00:09.608000Z,4,2,True,True,True,True
20,25,KR,Republic of Korea,2026-10-03T16:03:48.134000Z,4,2,True,True,True,True


In [ ]:
# Check whether the 100 returned countries represent the complete source result
countries_payload["meta"]

{'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 100, 'found': 158}

### Retrieve the Complete Country Set

The first country request returned 100 records while OpenAQ reports 158
available countries.

To avoid selecting warehouse countries from an incomplete sample, we now use
the existing paginated API client to retrieve every country page.

This also verifies that the reusable client behaves correctly against a real
multi-page metadata resource before the logic is used in production ingestion.

In [ ]:
# Collect every country page using the pagination logic already implemented in the API client
countries = [
    country
    for page in client.paginate(
        "countries",
        page_size=100,
    )
    for country in page
]

print(f"Countries retrieved: {len(countries)}")

Countries retrieved: 158


In [ ]:
# Rebuild the exploration dataset from the complete country population
countries_df = pd.DataFrame(
    [
        {
            "country_id": country["id"],
            "country_code": country["code"],
            "country_name": country["name"],
            "datetime_first": country.get("datetimeFirst"),
            "datetime_last": country.get("datetimeLast"),
            "parameters": [
                parameter["name"]
                for parameter in (country.get("parameters") or [])
            ],
            "parameter_count": len(
                country.get("parameters") or []
            ),
        }
        for country in countries
    ]
)

print(f"Countries in DataFrame: {len(countries_df)}")
countries_df.head()

Countries in DataFrame: 158


,country_id,country_code,country_name,datetime_first,datetime_last,parameters,parameter_count
0,1,ID,Indonesia,2016-01-30T01:00:00Z,2026-10-03T16:44:00Z,"[pm10, pm25, o3, o3, bc, no2, pm1, co2, wind_d...",20
1,2,MY,Malaysia,2022-11-03T21:00:00Z,2026-10-03T16:00:00Z,"[pm10, pm25, pm1, relativehumidity, temperatur...",6
2,3,CL,Chile,2016-01-30T01:00:00Z,2026-10-03T16:00:00Z,"[pm10, pm25, o3, co, no2, so2, pm1, relativehu...",10
3,4,BO,Bolivia,2026-06-22T20:00:00Z,2026-10-03T16:00:00Z,"[pm25, pm1, relativehumidity, temperature, um003]",5
4,5,PE,Peru,2016-03-04T17:00:00Z,2026-10-03T16:00:00Z,"[pm10, pm25, o3, co, no2, so2, no2, co, so2, o...",17


## Candidate Location Profiling

Country-level metadata confirms that Indonesia and India contain all four
target pollutants and all four environmental parameter types.

However, country-level availability does not guarantee that these parameters
coexist at individual monitoring locations.

We therefore inspect locations within the strongest candidate countries and
profile their sensor composition, units and observation periods. This will
identify locations capable of supporting pollutant trends, environmental
analysis and sensor-reliability metrics at compatible geographic grains.

In [ ]:
# Start with the two countries showing the strongest parameter coverage
CANDIDATE_COUNTRIES = {
    1: "Indonesia",
    9: "India",
}

candidate_locations = []

for country_id, country_name in CANDIDATE_COUNTRIES.items():
    # Retrieve every location for the country using the tested pagination logic
    for page in client.paginate(
        "locations",
        params={"countries_id": country_id},
        page_size=100,
    ):
        for location in page:
            candidate_locations.append(
                {
                    **location,
                    "_country_name": country_name,
                }
            )

print(f"Candidate locations retrieved: {len(candidate_locations)}")

Candidate locations retrieved: 846


### Inspect Location Structure Before Profiling

Before building the location comparison table, we inspect one complete location
record from the candidate population.

This confirms that the fields observed for R K Puram are representative of the
broader location source and helps identify nullable or inconsistent fields
before transformation logic is written.

In [ ]:
# Inspect the complete source structure before deciding which fields to flatten
candidate_locations[0]

{'id': 2537,
 'name': 'US Diplomatic Post: Jakarta South',
 'locality': None,
 'timezone': 'Asia/Jakarta',
 'country': {'id': 1, 'code': 'ID', 'name': 'Indonesia'},
 'owner': {'id': 4, 'name': 'Unknown Governmental Organization'},
 'provider': {'id': 257, 'name': 'StateAir JakartaSouth'},
 'isMobile': False,
 'isMonitor': True,
 'instruments': [{'id': 2, 'name': 'Government Monitor'}],
 'sensors': [{'id': 5047,
   'name': 'pm25 µg/m³',
   'parameter': {'id': 2,
    'name': 'pm25',
    'units': 'µg/m³',
    'displayName': 'PM2.5'}}],
 'coordinates': {'latitude': -6.236585, 'longitude': 106.793335},
 'licenses': None,
 'bounds': [106.793335, -6.236585, 106.793335, -6.236585],
 'distance': None,
 'datetimeFirst': {'utc': '2016-01-30T01:00:00Z',
  'local': '2016-01-30T08:00:00+07:00'},
 'datetimeLast': {'utc': '2016-11-09T16:00:00Z',
  'local': '2016-11-09T23:00:00+07:00'},
 '_country_name': 'Indonesia'}

### Build Location Sensor Profiles

Each location is summarized by its available sensor parameters rather than
measurement values.

The profile helps identify locations where the target pollutants and
environmental measurements coexist. Parameter units are retained because
matching parameter names alone does not guarantee analytical comparability.

**Source findings**

- Location metadata provides the country, owner and provider relationships
  required by the dimensional model.
- Sensors nested within locations provide the location-to-sensor relationship.
- Parameter names alone are insufficient for comparison because units can
  differ between sensors.
- Location observation timestamps can be null, so downstream schemas must
  allow missing source coverage dates.
- Country-level parameter availability does not imply that those parameters
  coexist at the same monitoring location.

In [ ]:
location_profiles = []

for location in candidate_locations:
    sensors = location.get("sensors") or []

    # Keep parameter and unit together so incompatible measurements remain visible
    parameter_units = {
        (
            sensor["parameter"]["name"],
            sensor["parameter"]["units"],
        )
        for sensor in sensors
        if sensor.get("parameter")
    }

    parameter_names = {
        name
        for name, _ in parameter_units
    }

    location_profiles.append(
        {
            "location_id": location["id"],
            "location_name": location["name"],
            "country": location["_country_name"],
            # Use UTC timestamps for consistent cross-country coverage comparison
            "datetime_first": (
                location.get("datetimeFirst") or {}
            ).get("utc"),
            "datetime_last": (
                location.get("datetimeLast") or {}
            ).get("utc"),
            "sensor_count": len(sensors),
            "core_pollutants": len(
                parameter_names & CORE_POLLUTANTS
            ),
            "weather_parameters": len(
                parameter_names & WEATHER_PARAMETERS
            ),
            "parameters": sorted(parameter_names),
            "parameter_units": sorted(parameter_units),
        }
)

locations_df = pd.DataFrame(location_profiles)

print(f"Locations profiled: {len(locations_df)}")
locations_df.head()

Locations profiled: 846


,location_id,location_name,country,datetime_first,datetime_last,sensor_count,core_pollutants,weather_parameters,parameters,parameter_units
0,2537,US Diplomatic Post: Jakarta South,Indonesia,2016-01-30T01:00:00Z,2016-11-09T16:00:00Z,1,1,0,[pm25],"[(pm25, µg/m³)]"
1,2538,US Diplomatic Post: Jakarta Central,Indonesia,2016-01-30T01:00:00Z,2016-11-09T16:00:00Z,1,1,0,[pm25],"[(pm25, µg/m³)]"
2,8320,Jakarta South,Indonesia,2022-10-03T20:00:00Z,2025-11-04T21:00:00Z,1,1,0,[pm25],"[(pm25, µg/m³)]"
3,8637,Jakarta Central,Indonesia,2016-11-09T18:00:00Z,2025-04-09T14:00:00Z,2,2,0,"[o3, pm25]","[(o3, ppm), (pm25, µg/m³)]"
4,223262,Cilandek,Indonesia,None,None,1,1,0,[pm25],"[(pm25, µg/m³)]"


In [ ]:
# Rank locations by useful pollutant/weather coverage, then sensor availability
location_candidates_df = (
    locations_df
    .sort_values(
        [
            "core_pollutants",
            "weather_parameters",
            "sensor_count",
        ],
        ascending=False,
    )
)

location_candidates_df[
    [
        "location_id",
        "location_name",
        "country",
        "datetime_last",
        "sensor_count",
        "core_pollutants",
        "weather_parameters",
        "parameter_units",
    ]
].head(20)

,location_id,location_name,country,datetime_last,sensor_count,core_pollutants,weather_parameters,parameter_units
458,229728,"Kamalnath Nagar, Bettiah - BSPCB",India,2026-10-02T08:30:00Z,19,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
86,17,"R K Puram, Delhi - DPCC",India,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
87,50,"Punjabi Bagh, Delhi - DPCC",India,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
89,235,"Anand Vihar, New Delhi - DPCC",India,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
92,301,"Vikas Sadan, Gurugram - HSPCB",India,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
96,407,"Zoo Park, Hyderabad - TSPCB",India,2026-10-01T04:00:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
105,860,"Sanjay Palace, Agra - UPPCB",India,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
109,2456,"Talkatora District Industries Center, Lucknow ...",India,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
118,2586,"Manali, Chennai - CPCB",India,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
131,5443,"Mahakaleshwar Temple, Ujjain - MPPCB",India,2025-05-05T06:00:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."


### Compare Strong Locations Within Each Country

The global ranking is dominated by Indian monitoring stations because many
locations contain all four target pollutants and all four environmental
parameter types.

A global top-N therefore does not provide the geographic diversity required
for country-level analysis.

We now rank locations within each country separately. This preserves geographic
diversity while allowing us to evaluate the strongest monitoring locations
available within each candidate country.

**Additional source-quality finding:** Some location metadata can contain
timestamps that appear ahead of the current extraction date. Source timestamps
will be preserved unchanged in Raw, while temporal validity checks will be
applied in downstream quality processing.

In [ ]:
# Show the strongest locations within each country rather than allowing one
# country's large monitoring network to dominate the candidate list
top_locations_by_country = (
    location_candidates_df
    .groupby("country", group_keys=False)
    .head(10)
)

top_locations_by_country[
    [
        "location_id",
        "location_name",
        "country",
        "datetime_first",
        "datetime_last",
        "sensor_count",
        "core_pollutants",
        "weather_parameters",
        "parameter_units",
    ]
]

,location_id,location_name,country,datetime_first,datetime_last,sensor_count,core_pollutants,weather_parameters,parameter_units
458,229728,"Kamalnath Nagar, Bettiah - BSPCB",India,2021-07-17T22:00:00Z,2026-10-02T08:30:00Z,19,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
86,17,"R K Puram, Delhi - DPCC",India,2016-02-05T14:55:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
87,50,"Punjabi Bagh, Delhi - DPCC",India,2016-02-05T14:55:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
89,235,"Anand Vihar, New Delhi - DPCC",India,2016-02-05T14:15:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
92,301,"Vikas Sadan, Gurugram - HSPCB",India,2016-03-25T08:00:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
96,407,"Zoo Park, Hyderabad - TSPCB",India,2016-03-21T19:00:00Z,2026-10-01T04:00:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
105,860,"Sanjay Palace, Agra - UPPCB",India,2016-03-22T01:00:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
109,2456,"Talkatora District Industries Center, Lucknow ...",India,2016-03-21T10:00:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
118,2586,"Manali, Chennai - CPCB",India,2016-03-21T10:00:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
131,5443,"Mahakaleshwar Temple, Ujjain - MPPCB",India,2017-11-20T06:00:00Z,2025-05-05T06:00:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."


## Expand Geographic Candidate Search

The first location-level comparison shows that country-level parameter
availability does not guarantee equivalent coverage at individual monitoring
locations.

India contains many locations with all four target pollutants and all four
environmental parameter types, while the strongest Indonesian locations have
substantially narrower coverage.

We therefore retain India as a strong candidate and expand location profiling
to additional countries with recent data and complete core-pollutant coverage.
The goal is to identify approximately three countries with comparable,
analytically useful monitoring locations rather than forcing countries into
the warehouse based on country-level metadata alone.

In [ ]:
# Countries retained for deeper location-level investigation
CANDIDATE_CODES = {
    "IN",
    "CL",
    "PE",
    "AR",
    "ZA",
    "FR",
    "DE",
}

candidate_country_lookup = (
    countries_df[
        countries_df["country_code"].isin(CANDIDATE_CODES)
    ]
    [["country_id", "country_code", "country_name"]]
    .sort_values("country_name")
)

candidate_country_lookup

,country_id,country_code,country_name
5,6,AR,Argentina
2,3,CL,Chile
18,22,FR,France
39,50,DE,Germany
8,9,IN,India
4,5,PE,Peru
27,37,ZA,South Africa


### Retrieve Expanded Candidate Locations

We now retrieve location metadata for the reduced candidate-country pool.

The same location-level profiling criteria will be applied across every
country so that geographic selection is based on comparable evidence rather
than different rules for different countries.

In [ ]:
candidate_locations = []

for row in candidate_country_lookup.itertuples(index=False):
    # Retrieve every monitoring location reported for each candidate country
    for page in client.paginate(
        "locations",
        params={"countries_id": row.country_id},
        page_size=100,
    ):
        for location in page:
            candidate_locations.append(
                {
                    **location,
                    "_country_name": row.country_name,
                    "_country_code": row.country_code,
                }
            )

print(f"Candidate locations retrieved: {len(candidate_locations)}")

Candidate locations retrieved: 3125


In [ ]:
# Compare the five strongest monitoring locations within each candidate country
top_locations_by_country = (
    location_candidates_df
    .groupby("country", group_keys=False)
    .head(5)
)

top_locations_by_country[
    [
        "location_id",
        "location_name",
        "country",
        "datetime_first",
        "datetime_last",
        "sensor_count",
        "core_pollutants",
        "weather_parameters",
        "parameter_units",
    ]
]

,location_id,location_name,country,datetime_first,datetime_last,sensor_count,core_pollutants,weather_parameters,parameter_units
458,229728,"Kamalnath Nagar, Bettiah - BSPCB",India,2021-07-17T22:00:00Z,2026-10-02T08:30:00Z,19,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
86,17,"R K Puram, Delhi - DPCC",India,2016-02-05T14:55:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
87,50,"Punjabi Bagh, Delhi - DPCC",India,2016-02-05T14:55:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
89,235,"Anand Vihar, New Delhi - DPCC",India,2016-02-05T14:15:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
92,301,"Vikas Sadan, Gurugram - HSPCB",India,2016-03-25T08:00:00Z,2026-10-02T08:30:00Z,18,4,4,"[(co, ppb), (co, µg/m³), (no, ppb), (no2, ppb)..."
7,1367443,Depok,Indonesia,2023-08-01T07:00:00Z,2025-12-20T07:00:00Z,8,2,2,"[(pm1, µg/m³), (pm10, µg/m³), (pm25, µg/m³), (..."
9,1563313,Qoryah Darussalam,Indonesia,2023-09-12T10:00:00Z,2025-10-17T17:00:00Z,8,2,2,"[(pm1, µg/m³), (pm10, µg/m³), (pm25, µg/m³), (..."
10,1776543,Jihan Fauziah Hamdi,Indonesia,2023-10-30T13:00:00Z,2024-05-26T00:00:00Z,8,2,2,"[(pm1, µg/m³), (pm10, µg/m³), (pm25, µg/m³), (..."
13,2071976,Krukut,Indonesia,2024-01-09T08:00:00Z,2024-09-10T08:00:00Z,8,2,2,"[(pm1, µg/m³), (pm10, µg/m³), (pm25, µg/m³), (..."
14,2849176,Margonda Residence 3,Indonesia,2024-05-16T10:00:00Z,2024-09-04T05:00:00Z,6,2,2,"[(pm1, µg/m³), (pm10, µg/m³), (pm25, µg/m³), (..."


In [ ]:
location_profiles = []

for location in candidate_locations:
    sensors = location.get("sensors") or []

    # Keep parameter and unit together so incompatible measurements stay visible
    parameter_units = {
        (
            sensor["parameter"]["name"],
            sensor["parameter"]["units"],
        )
        for sensor in sensors
        if sensor.get("parameter")
    }

    parameter_names = {
        name
        for name, _ in parameter_units
    }

    location_profiles.append(
        {
            "location_id": location["id"],
            "location_name": location["name"],
            "country": location["_country_name"],
            # Use UTC timestamps for consistent cross-country coverage comparison
            "datetime_first": (
                location.get("datetimeFirst") or {}
            ).get("utc"),
            "datetime_last": (
                location.get("datetimeLast") or {}
            ).get("utc"),
            "sensor_count": len(sensors),
            "core_pollutants": len(
                parameter_names & CORE_POLLUTANTS
            ),
            "weather_parameters": len(
                parameter_names & WEATHER_PARAMETERS
            ),
            "parameters": sorted(parameter_names),
            "parameter_units": sorted(parameter_units),
        }
    )

# Rebuild from the current candidate-location population
locations_df = pd.DataFrame(location_profiles)

print(f"Locations profiled: {len(locations_df)}")
print(f"Countries profiled: {locations_df['country'].nunique()}")
print(sorted(locations_df["country"].unique()))

Locations profiled: 3125
Countries profiled: 7
['Argentina', 'Chile', 'France', 'Germany', 'India', 'Peru', 'South Africa']


### Compare Top Locations Across Candidate Countries

The refreshed location dataset contains all seven candidate countries, but a
global ranking is dominated by India because its monitoring network contains
many locations with broad parameter coverage.

To compare countries fairly during dataset selection, we retain the five
strongest locations within each country. This is a selection aid only; the
final warehouse scope will also require sensor-level lifecycle and unit
compatibility checks.

In [ ]:
# Rank the full seven-country population before selecting the best locations per country
location_candidates_df = locations_df.sort_values(
    [
        "core_pollutants",
        "weather_parameters",
        "sensor_count",
    ],
    ascending=False,
)

# Keep the five strongest locations from each country for a balanced comparison
top_locations_by_country = (
    location_candidates_df
    .groupby("country", sort=True, group_keys=False)
    .head(5)
    .sort_values(
        [
            "country",
            "core_pollutants",
            "weather_parameters",
            "sensor_count",
        ],
        ascending=[True, False, False, False],
    )
)

print(
    top_locations_by_country
    .groupby("country")
    .size()
)

top_locations_by_country[
    [
        "location_id",
        "location_name",
        "country",
        "datetime_first",
        "datetime_last",
        "sensor_count",
        "core_pollutants",
        "weather_parameters",
        "parameter_units",
    ]
]

country
Argentina       5
Chile           5
France          5
Germany         5
India           5
Peru            5
South Africa    5
dtype: int64


,location_id,location_name,country,datetime_first,datetime_last,sensor_count,core_pollutants,weather_parameters,parameter_units
5,1212895,EMC I Dock Sud,Argentina,2023-07-13T13:00:00Z,2024-06-03T12:00:00Z,8,4,0,"[(co, µg/m³), (no, µg/m³), (no2, µg/m³), (nox,..."
6,1212896,EMC II La Matanza,Argentina,2023-07-13T13:00:00Z,2024-06-03T11:00:00Z,8,4,0,"[(co, µg/m³), (no, µg/m³), (no2, µg/m³), (nox,..."
11,1658093,saladillo_urban_1,Argentina,2023-10-04T03:00:00Z,2024-04-19T18:00:00Z,8,2,2,"[(pm1, µg/m³), (pm10, µg/m³), (pm25, µg/m³), (..."
14,2099180,"La Matanza, Buenos Aires",Argentina,2024-01-15T15:00:00Z,2026-10-03T16:00:00Z,6,2,2,"[(pm1, µg/m³), (pm10, µg/m³), (pm25, µg/m³), (..."
15,2798193,EET1_Saladillo,Argentina,2024-04-21T13:00:00Z,2026-10-03T13:00:00Z,6,2,2,"[(pm1, µg/m³), (pm10, µg/m³), (pm25, µg/m³), (..."
48,25,Parque O'Higgins,Chile,2016-01-30T01:00:00Z,2026-10-03T14:00:00Z,6,4,0,"[(co, µg/m³), (no2, µg/m³), (o3, µg/m³), (pm10..."
50,27,Concón,Chile,2016-01-30T01:00:00Z,2026-10-03T13:00:00Z,6,4,0,"[(co, µg/m³), (no2, µg/m³), (o3, µg/m³), (pm10..."
53,45,Puente Alto,Chile,2016-01-30T01:00:00Z,2026-10-03T13:00:00Z,6,4,0,"[(co, µg/m³), (no2, µg/m³), (o3, µg/m³), (pm10..."
54,47,Valparaiso,Chile,2016-01-30T01:00:00Z,2026-10-03T14:00:00Z,6,4,0,"[(co, µg/m³), (no2, µg/m³), (o3, µg/m³), (pm10..."
55,54,Talagante,Chile,2016-01-30T01:00:00Z,2026-10-03T13:00:00Z,6,4,0,"[(co, µg/m³), (no2, µg/m³), (o3, µg/m³), (pm10..."


## Sensor-Level Comparability Investigation

Location-level metadata identifies several strong candidates, but location
coverage does not prove that individual pollutant sensors operated during the
same historical period.

We now inspect the sensor inventory of representative finalist locations.

The goal is to verify:

- individual sensor lifecycles
- pollutant units
- replacement or legacy sensors
- overlapping observation periods
- suitability for cross-country comparison

This is the final source-selection gate before the warehouse's geographic and
historical scope is frozen.

In [ ]:
# Representative strong locations selected for sensor-level investigation
FINALIST_LOCATIONS = {
    17: ("India", "R K Puram, Delhi - DPCC"),
    2669: ("Germany", "München/Stachus"),
    3549: ("France", "St-Laurent-Blangy"),
    225464: ("South Africa", "Karoo-NAQI"),
}

finalist_sensors = []

for location_id, (country, location_name) in FINALIST_LOCATIONS.items():
    location = client.get(
        f"locations/{location_id}"
    )["results"][0]

    # Preserve each physical sensor separately because sensors can be
    # replaced while measuring the same parameter at one location.
    for sensor in location.get("sensors") or []:
        finalist_sensors.append(
            {
                "country": country,
                "location_id": location_id,
                "location_name": location_name,
                "sensor_id": sensor["id"],
                "sensor_name": sensor["name"],
                "parameter_id": sensor["parameter"]["id"],
                "parameter": sensor["parameter"]["name"],
                "unit": sensor["parameter"]["units"],
            }
        )

finalist_sensors_df = pd.DataFrame(finalist_sensors)

print(
    finalist_sensors_df.groupby("country")["sensor_id"].count()
)

finalist_sensors_df.sort_values(
    ["country", "parameter", "sensor_id"]
)

country
France           7
Germany          7
India           18
South Africa    13
Name: sensor_id, dtype: int64


,country,location_id,location_name,sensor_id,sensor_name,parameter_id,parameter,unit
25,France,3549,St-Laurent-Blangy,7868,co µg/m³,4,co,µg/m³
26,France,3549,St-Laurent-Blangy,4274385,no µg/m³,19843,no,µg/m³
27,France,3549,St-Laurent-Blangy,8264,no2 µg/m³,5,no2,µg/m³
28,France,3549,St-Laurent-Blangy,8263,o3 µg/m³,3,o3,µg/m³
29,France,3549,St-Laurent-Blangy,7951,pm10 µg/m³,1,pm10,µg/m³
30,France,3549,St-Laurent-Blangy,9014237,pm25 µg/m³,2,pm25,µg/m³
31,France,3549,St-Laurent-Blangy,7860,so2 µg/m³,6,so2,µg/m³
18,Germany,2669,München/Stachus,7559,co µg/m³,4,co,µg/m³
19,Germany,2669,München/Stachus,4275101,no µg/m³,19843,no,µg/m³
20,Germany,2669,München/Stachus,6681,no2 µg/m³,5,no2,µg/m³


### Validate Comparable Sensor Lifecycles

PM2.5 and PM10 use the same `µg/m³` unit across all four finalist countries,
making them strong candidates for cross-country analysis.

Location-level dates are insufficient because individual sensors may be
replaced or operate during different periods.

We therefore inspect the lifecycle of each PM2.5 and PM10 sensor to determine
a common historical window and distinguish legacy sensors from current
replacements before finalizing the warehouse backfill scope.

In [ ]:
# PM2.5 and PM10 currently provide direct unit comparability across finalists
COMPARABLE_PARAMETERS = {"pm25", "pm10"}

comparable_sensors = finalist_sensors_df[
    finalist_sensors_df["parameter"].isin(COMPARABLE_PARAMETERS)
].copy()

sensor_lifecycles = []

for sensor in comparable_sensors.itertuples(index=False):
    payload = client.get(
        f"sensors/{sensor.sensor_id}"
    )["results"][0]

    sensor_lifecycles.append(
        {
            "country": sensor.country,
            "location_id": sensor.location_id,
            "location_name": sensor.location_name,
            "sensor_id": sensor.sensor_id,
            "parameter": sensor.parameter,
            "unit": sensor.unit,
            # Sensor-level dates determine actual historical comparability
            "datetime_first": (
                payload.get("datetimeFirst") or {}
            ).get("utc"),
            "datetime_last": (
                payload.get("datetimeLast") or {}
            ).get("utc"),
        }
    )

sensor_lifecycles_df = pd.DataFrame(sensor_lifecycles)

sensor_lifecycles_df.sort_values(
    ["country", "parameter", "datetime_first"]
)

,country,location_id,location_name,sensor_id,parameter,unit,datetime_first,datetime_last
6,France,3549,St-Laurent-Blangy,7951,pm10,µg/m³,2017-02-18T08:00:00Z,2026-10-03T15:00:00Z
7,France,3549,St-Laurent-Blangy,9014237,pm25,µg/m³,2024-04-27T20:00:00Z,2026-10-03T15:00:00Z
4,Germany,2669,München/Stachus,6686,pm10,µg/m³,2016-11-21T11:00:00Z,2026-10-03T15:00:00Z
5,Germany,2669,München/Stachus,6688,pm25,µg/m³,2016-11-21T15:00:00Z,2026-10-03T15:00:00Z
1,India,17,"R K Puram, Delhi - DPCC",399,pm10,µg/m³,2016-02-05T14:55:00Z,2018-02-21T21:15:00Z
0,India,17,"R K Puram, Delhi - DPCC",12234786,pm10,µg/m³,2025-02-18T20:30:00Z,2026-10-02T08:30:00Z
2,India,17,"R K Puram, Delhi - DPCC",35,pm25,µg/m³,2016-02-05T14:55:00Z,2018-02-21T21:15:00Z
3,India,17,"R K Puram, Delhi - DPCC",12234787,pm25,µg/m³,2025-02-18T20:15:00Z,2026-10-02T08:30:00Z
8,South Africa,225464,Karoo-NAQI,1303615,pm10,µg/m³,2021-04-29T17:00:00Z,2026-10-03T16:00:00Z
9,South Africa,225464,Karoo-NAQI,1303537,pm25,µg/m³,2021-04-29T17:00:00Z,2026-10-03T16:00:00Z


## Source Selection Decision

Source investigation identified four locations suitable for the planned
warehouse model:

- R K Puram, Delhi - DPCC (India)
- München/Stachus (Germany)
- St-Laurent-Blangy (France)
- Karoo-NAQI (South Africa)

PM2.5 and PM10 are available in `µg/m³` across all four locations and therefore
form the initial basis for direct cross-country air-quality comparison.

Sensor lifecycle analysis shows that the common comparable period begins in
February 2025. R K Puram's current PM2.5 and PM10 sensors began operating on
2025-02-18, while the equivalent sensors in Germany, France and South Africa
were already active.

R K Puram also contains legacy PM2.5 and PM10 sensors that ended in 2018.
These remain separate sensor entities and must not be merged with their 2025
replacements.

The initial cross-country analytical window is therefore:

**2025-02-18 to 2026-09-29**

Raw ingestion may retain broader source history and additional parameters.
The common date range and pollutant restrictions apply specifically to
analytics requiring direct cross-country comparability.

R K Puram also provides temperature, relative humidity, wind speed and wind
direction sensors. These support a separate environmental-conditions analysis
without requiring meteorological measurements to exist at every international
location.

## Investigate Parameter Source Completeness

The Silver relationship check revealed that sensors reference parameter IDs
`1`, `2`, `22`, `34`, `98`, and `100`, while Raw contains metadata only for
parameter ID `1`.

Before changing ingestion code, inspect the OpenAQ parameter source directly.
The goal is to determine whether the missing parameters are available from the
source and whether our ingestion scope, rather than the API itself, caused the
incomplete Raw dataset.

In [ ]:
# Query the parameter resource directly to inspect what OpenAQ currently exposes
parameter_payload = client.get("parameters")

parameter_payload

{'meta': {'name': 'openaq-api',
  'website': '/',
  'page': 1,
  'limit': 100,
  'found': 43},
 'results': [{'id': 1,
   'name': 'pm10',
   'units': 'µg/m³',
   'displayName': 'PM10',
   'description': 'Particulate matter less than 10 micrometers in diameter mass concentration'},
  {'id': 2,
   'name': 'pm25',
   'units': 'µg/m³',
   'displayName': 'PM2.5',
   'description': 'Particulate matter less than 2.5 micrometers in diameter mass concentration'},
  {'id': 3,
   'name': 'o3',
   'units': 'µg/m³',
   'displayName': 'O₃ mass',
   'description': 'Ozone mass concentration'},
  {'id': 4,
   'name': 'co',
   'units': 'µg/m³',
   'displayName': 'CO mass',
   'description': 'Carbon Monoxide mass concentration'},
  {'id': 5,
   'name': 'no2',
   'units': 'µg/m³',
   'displayName': 'NO₂ mass',
   'description': 'Nitrogen Dioxide mass concentration'},
  {'id': 6,
   'name': 'so2',
   'units': 'µg/m³',
   'displayName': 'SO₂ mass',
   'description': 'Sulfur Dioxide mass concentration'},
  {'

In [ ]:
# Extract the returned parameter records for easier inspection
parameter_records = parameter_payload.get("results", [])

print(f"Parameters returned: {len(parameter_records)}")

[
    {
        "id": parameter.get("id"),
        "name": parameter.get("name"),
        "display_name": parameter.get("displayName"),
        "units": parameter.get("units"),
    }
    for parameter in parameter_records
]

Parameters returned: 43


[{'id': 1, 'name': 'pm10', 'display_name': 'PM10', 'units': 'µg/m³'},
 {'id': 2, 'name': 'pm25', 'display_name': 'PM2.5', 'units': 'µg/m³'},
 {'id': 3, 'name': 'o3', 'display_name': 'O₃ mass', 'units': 'µg/m³'},
 {'id': 4, 'name': 'co', 'display_name': 'CO mass', 'units': 'µg/m³'},
 {'id': 5, 'name': 'no2', 'display_name': 'NO₂ mass', 'units': 'µg/m³'},
 {'id': 6, 'name': 'so2', 'display_name': 'SO₂ mass', 'units': 'µg/m³'},
 {'id': 7, 'name': 'no2', 'display_name': 'NO₂', 'units': 'ppm'},
 {'id': 8, 'name': 'co', 'display_name': 'CO', 'units': 'ppm'},
 {'id': 9, 'name': 'so2', 'display_name': 'SO₂', 'units': 'ppm'},
 {'id': 10, 'name': 'o3', 'display_name': 'O₃', 'units': 'ppm'},
 {'id': 11, 'name': 'bc', 'display_name': 'BC', 'units': 'µg/m³'},
 {'id': 15, 'name': 'no2', 'display_name': 'NO₂', 'units': 'ppb'},
 {'id': 19, 'name': 'pm1', 'display_name': 'PM1', 'units': 'µg/m³'},
 {'id': 21, 'name': 'co2', 'display_name': 'CO₂', 'units': 'ppm'},
 {'id': 22,
  'name': 'wind_direction',


In [ ]:
# These are the parameter IDs referenced by the sensors in our current dataset
required_parameter_ids = {1, 2, 22, 34, 98, 100}

required_parameters = [
    parameter
    for parameter in parameter_records
    if parameter.get("id") in required_parameter_ids
]

required_parameters

[{'id': 1,
  'name': 'pm10',
  'units': 'µg/m³',
  'displayName': 'PM10',
  'description': 'Particulate matter less than 10 micrometers in diameter mass concentration'},
 {'id': 2,
  'name': 'pm25',
  'units': 'µg/m³',
  'displayName': 'PM2.5',
  'description': 'Particulate matter less than 2.5 micrometers in diameter mass concentration'},
 {'id': 22,
  'name': 'wind_direction',
  'units': 'deg',
  'displayName': 'Wind direction',
  'description': 'Direction that the wind originates from'},
 {'id': 34,
  'name': 'wind_speed',
  'units': 'm/s',
  'displayName': 'Wind speed',
  'description': 'Average wind speed in meters per second'},
 {'id': 98,
  'name': 'relativehumidity',
  'units': '%',
  'displayName': 'RH',
  'description': None},
 {'id': 100,
  'name': 'temperature',
  'units': 'c',
  'displayName': 'Temperature (C)',
  'description': None}]